# WebMD -> Qdrant Cloud Incremental Ingestion (Colab)

Colab-ready notebook variant using Qdrant Cloud URL and API key.
This does not modify your existing local Python scripts.


In [ ]:
!pip -q install qdrant-client sentence-transformers numpy tqdm


## Optional: Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configure Qdrant Cloud and Paths


In [ ]:
import os

os.environ['QDRANT_URL'] = '<YOUR_QDRANT_URL>'
# '<YOUR_QDRANT_URL>'
os.environ['QDRANT_API_KEY'] = '<YOUR_QDRANT_API_KEY>'
# '<YOUR_QDRANT_API_KEY>'

os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/MisInformation_Research/data_scraping'

os.environ['COLLECTION_NAME'] = 'medical_facts_global_1'
os.environ['FINE_TUNE_MODEL_PATH'] = '/content/drive/MyDrive/MisInformation_Research/models/fine_tuned_medical_retriever'
os.environ['BATCH_SIZE'] = '8' # Reduced batch size to mitigate OOM error
os.environ['UPSERT_BATCH_SIZE'] = '64'
os.environ['DELETE_BATCH_SIZE'] = '128'

# Memory controls for large runs
os.environ['EMBED_WRITE_CHUNK_SIZE'] = '512'   # embed+upsert passages per loop
os.environ['MAX_PASSAGES_PER_DOC'] = '40'      # 0 means no per-doc cap


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from hashlib import sha1
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointIdsList, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

PROJECT_ROOT = Path(os.getenv('PROJECT_ROOT', '/content')).resolve()
INPUT_SOURCES = [
    {'path': 'storage/webmd/articles', 'source_type': 'article'},
    {'path': 'storage/webmd/healthtopics', 'source_type': 'health_topic'},
]
FILE_GLOB = '*.json'
SOURCE_SITE = 'webmd'

QDRANT_URL = os.getenv('QDRANT_URL', '').strip()
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY', '').strip() or None
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'medical_facts_global_1')

SAPBERT_MODEL = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
FINE_TUNE_MODEL_PATH_SETTING = os.getenv('FINE_TUNE_MODEL_PATH', 'storage/models/fine_tuned_medical_retriever')
FINE_TUNE_MODEL_PATH = Path(FINE_TUNE_MODEL_PATH_SETTING)
if not FINE_TUNE_MODEL_PATH.is_absolute():
    FINE_TUNE_MODEL_PATH = (PROJECT_ROOT / FINE_TUNE_MODEL_PATH).resolve()

BATCH_SIZE = int(os.getenv('BATCH_SIZE', '16'))
UPSERT_BATCH_SIZE = int(os.getenv('UPSERT_BATCH_SIZE', '64'))
DELETE_BATCH_SIZE = int(os.getenv('DELETE_BATCH_SIZE', '128'))
NORMALIZE_EMBEDDINGS = True

DELETE_OLD_PASSAGES_FOR_UPDATED_DOC = True
DELETE_REMOVED_DOCS = True
FORCE_REPROCESS_ALL_DOCS = False

STATE_FILE = 'storage/outputs/webmd/qdrant_ingest_state_colab_global_1.json'

if not QDRANT_URL:
    raise ValueError('Set QDRANT_URL for Qdrant Cloud endpoint before running.')
if 'localhost' in QDRANT_URL or '127.0.0.1' in QDRANT_URL:
    raise ValueError('QDRANT_URL points to localhost; use your Qdrant Cloud URL in Colab.')
if not FINE_TUNE_MODEL_PATH.exists():
    raise FileNotFoundError(f'Fine-tuned model path not found: {FINE_TUNE_MODEL_PATH}')

EMBEDDING_MODEL = str(FINE_TUNE_MODEL_PATH)  # Qdrant named vector: primary


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')


def sha1_hex(value: str) -> str:
    return sha1(value.encode('utf-8')).hexdigest()


def normalize_space(text: str) -> str:
    if not text:
        return ''
    return ' '.join(text.split()).strip()


def dedupe_keep_order(values: Iterable[str]) -> List[str]:
    out: List[str] = []
    seen = set()
    for value in values:
        if not value:
            continue
        if value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out


def canonical_hash(obj: Any) -> str:
    serialized = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return sha1_hex(serialized)


def load_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def normalize_to_doc_list(data: Any) -> List[Dict[str, Any]]:
    if isinstance(data, dict):
        if data.get('error'):
            return []
        if any(k in data for k in ('url', 'title', 'sections', 'content_blocks')):
            return [data]
        if isinstance(data.get('items'), list):
            return [x for x in data['items'] if isinstance(x, dict)]
        return [v for v in data.values() if isinstance(v, dict)]

    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]

    return []


def make_doc_uid(source_site: str, source_type: str, article: Dict[str, Any]) -> str:
    url = normalize_space(str(article.get('url') or ''))
    title = normalize_space(str(article.get('title') or ''))
    key = f'{source_site}|{source_type}|{url or title}'
    return sha1_hex(key)


def build_passages(article: Dict[str, Any], source_site: str, source_type: str, source_file: str):
    doc_uid = make_doc_uid(source_site, source_type, article)

    base_payload = {
        'source_site': source_site,
        'source_type': source_type,
        'source_file': source_file,
        'doc_id': doc_uid,
        'url': article.get('url'),
        'title': article.get('title'),
        'published_date': article.get('published_date'),
        'author': article.get('author'),
        'medically_reviewed_by': article.get('medically_reviewed_by'),
        'tags': article.get('tags'),
        'scrape_timestamp_utc': article.get('scrape_timestamp_utc'),
        'first_letter': article.get('first_letter'),
        'read_time': article.get('read_time'),
        'sources': article.get('sources'),
    }

    sections = article.get('sections') or []
    passages: List[Dict[str, Any]] = []
    point_ids: List[str] = []
    seen_texts = set()

    for section_index, section in enumerate(sections):
        if not isinstance(section, dict):
            continue

        section_heading = section.get('heading')
        content_blocks = section.get('content_blocks') or []

        fallback_content = section.get('content')
        if not content_blocks and isinstance(fallback_content, list):
            generated_blocks = []
            for item in fallback_content:
                if isinstance(item, str):
                    generated_blocks.append({'text': item, 'associated_bullets': None})
                elif isinstance(item, dict):
                    generated_blocks.append({'text': item.get('text'), 'associated_bullets': item.get('bullets') or item.get('associated_bullets')})
            content_blocks = generated_blocks

        for block_index, block in enumerate(content_blocks):
            if not isinstance(block, dict):
                continue

            paragraph_text = normalize_space(str(block.get('text') or ''))
            bullets = block.get('associated_bullets')
            if bullets is None:
                bullets = block.get('bullets')
            if bullets is None:
                bullets = []
            if not isinstance(bullets, list):
                bullets = [str(bullets)]
            bullets = [normalize_space(str(item)) for item in bullets if normalize_space(str(item))]

            if bullets:
                bullets_text = '\n'.join(bullets)
                full_text = (paragraph_text + '\n' + bullets_text).strip() if paragraph_text else bullets_text
            else:
                full_text = paragraph_text

            if not full_text:
                continue
            if full_text in seen_texts:
                continue
            seen_texts.add(full_text)

            passage_id = f'{doc_uid}_s{section_index}_b{block_index}'
            point_id = sha1_hex(f"{source_site}|{passage_id}|{article.get('url') or article.get('title') or ''}")

            payload = {
                **base_payload,
                'passage_id': passage_id,
                'section_heading': section_heading,
                'block_index': block_index,
                'text': full_text,
            }

            passages.append({'point_id': point_id, 'payload': payload, 'text': full_text})
            point_ids.append(point_id)

    return doc_uid, passages, dedupe_keep_order(point_ids)


def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10
    return vectors / norms


def load_state(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {'version': 1, 'source_site': SOURCE_SITE, 'docs': {}}
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, dict):
        return {'version': 1, 'source_site': SOURCE_SITE, 'docs': {}}
    data.setdefault('docs', {})
    return data


def save_state(path: Path, docs_state: Dict[str, Any], input_sources: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        'version': 1,
        'source_site': SOURCE_SITE,
        'updated_at_utc': utc_now_iso(),
        'input_sources': input_sources,
        'docs': docs_state,
    }
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def batched(items: List[Any], size: int) -> Iterable[List[Any]]:
    for i in range(0, len(items), size):
        yield items[i:i + size]


def ensure_collection(client: QdrantClient, primary_dim: int, sapbert_dim: int) -> None:
    if client.collection_exists(COLLECTION_NAME):
        print(f'[Qdrant] Collection exists: {COLLECTION_NAME}')
        return
    print(f'[Qdrant] Creating collection: {COLLECTION_NAME}')
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            'primary': VectorParams(size=primary_dim, distance=Distance.COSINE),
            'sapbert': VectorParams(size=sapbert_dim, distance=Distance.COSINE),
        },
    )


def delete_points(client: QdrantClient, point_ids: List[str]) -> int:
    if not point_ids:
        return 0
    unique_ids = dedupe_keep_order(point_ids)
    print(f'[Qdrant] Deleting {len(unique_ids)} stale points...')
    deleted = 0
    for chunk in tqdm(list(batched(unique_ids, DELETE_BATCH_SIZE)), desc='Delete batches'):
        client.delete(collection_name=COLLECTION_NAME, points_selector=PointIdsList(points=chunk), wait=True)
        deleted += len(chunk)
    return deleted


def embed_texts(model_name: str, texts: List[str], batch_size: int, normalize: bool) -> np.ndarray:
    print(f'[Embed] Loading model: {model_name}')
    model = SentenceTransformer(model_name)
    print(f'[Embed] Encoding {len(texts)} passages...')
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    if normalize:
        vectors = l2_normalize(vectors)
        print('[Embed] L2 normalization applied')
    return vectors


def upsert_points(
    client: QdrantClient,
    passages: List[Dict[str, Any]],
    primary_vectors: np.ndarray,
    sapbert_vectors: np.ndarray,
) -> int:
    total = len(passages)
    if total == 0:
        return 0

    all_points: List[PointStruct] = []
    for i, passage in enumerate(passages):
        all_points.append(PointStruct(
            id=passage['point_id'],
            vector={
                'primary': primary_vectors[i].astype(float).tolist(),
                'sapbert': sapbert_vectors[i].astype(float).tolist(),
            },
            payload=passage['payload'],
        ))

    print(f'[Qdrant] Upserting {total} points...')
    upserted = 0
    for chunk in tqdm(list(batched(all_points, UPSERT_BATCH_SIZE)), desc='Upsert batches'):
        client.upsert(collection_name=COLLECTION_NAME, points=chunk, wait=True)
        upserted += len(chunk)
    return upserted


def iter_source_docs(source_path: Path, file_glob: str):
    if source_path.is_file():
        data = load_json(source_path)
        for doc in normalize_to_doc_list(data):
            yield doc, source_path.name
        return

    if source_path.is_dir():
        json_files = sorted(source_path.glob(file_glob))
        for file_path in tqdm(json_files, desc=f'Read {source_path.name}'):
            try:
                data = load_json(file_path)
            except Exception as exc:
                print(f'  [Warn] Failed to parse {file_path.name}: {exc}')
                continue
            for doc in normalize_to_doc_list(data):
                yield doc, file_path.name
        return

    raise FileNotFoundError(f'Input source not found: {source_path}')


def main() -> None:
    state_path = (PROJECT_ROOT / STATE_FILE).resolve()

    print('=' * 72)
    print('WebMD -> Qdrant Cloud Incremental Ingestion (Colab)')
    print('=' * 72)
    print(f'[Config] Project root: {PROJECT_ROOT}')
    print(f'[Config] Qdrant URL: {QDRANT_URL}')
    print(f'[Config] Collection: {COLLECTION_NAME}')
    print(f'[Config] State file: {state_path}')

    source_paths: List[Tuple[Path, str]] = []
    for src in INPUT_SOURCES:
        rel_path = src.get('path')
        source_type = src.get('source_type') or 'unknown'
        abs_path = (PROJECT_ROOT / rel_path).resolve()
        if not abs_path.exists():
            print(f'  [Warn] Missing source, skipping: {abs_path}')
            continue
        source_paths.append((abs_path, source_type))
        print(f'  - {abs_path} ({source_type})')

    if not source_paths:
        raise RuntimeError('No valid WebMD input sources found under PROJECT_ROOT.')

    state = load_state(state_path)
    previous_docs: Dict[str, Any] = state.get('docs', {})

    current_docs: Dict[str, Any] = {}
    passages_to_upsert: List[Dict[str, Any]] = []
    stale_ids_to_delete: List[str] = []

    docs_seen = 0
    docs_skipped = 0
    docs_new = 0
    docs_changed = 0

    print('[Step 1/5] Scanning and diffing source documents...')
    for source_path, source_type in source_paths:
        for doc, source_file in iter_source_docs(source_path, FILE_GLOB):
            docs_seen += 1
            if not isinstance(doc, dict) or not (doc.get('url') or doc.get('title')):
                continue
            if doc.get('error'):
                continue

            doc_uid = make_doc_uid(SOURCE_SITE, source_type, doc)
            fingerprint = canonical_hash(doc)
            previous = previous_docs.get(doc_uid)

            is_unchanged = (
                not FORCE_REPROCESS_ALL_DOCS
                and previous is not None
                and previous.get('fingerprint') == fingerprint
                and previous.get('primary_model') == EMBEDDING_MODEL
                and previous.get('sapbert_model') == SAPBERT_MODEL
            )
            if is_unchanged:
                current_docs[doc_uid] = previous
                docs_skipped += 1
                continue

            _, passages, point_ids = build_passages(doc, SOURCE_SITE, source_type, source_file)

            if previous is None:
                docs_new += 1
            else:
                docs_changed += 1
                if DELETE_OLD_PASSAGES_FOR_UPDATED_DOC:
                    stale_ids_to_delete.extend(previous.get('point_ids') or [])

            current_docs[doc_uid] = {'fingerprint': fingerprint, 'point_ids': point_ids, 'source_type': source_type, 'primary_model': EMBEDDING_MODEL, 'sapbert_model': SAPBERT_MODEL, 'source_file': source_file, 'updated_at_utc': utc_now_iso()}
            passages_to_upsert.extend(passages)

    previous_doc_ids = set(previous_docs.keys())
    current_doc_ids = set(current_docs.keys())
    removed_doc_ids = sorted(previous_doc_ids - current_doc_ids)

    if removed_doc_ids and DELETE_REMOVED_DOCS:
        for doc_uid in removed_doc_ids:
            stale_ids_to_delete.extend(previous_docs.get(doc_uid, {}).get('point_ids') or [])

    print('[Step 2/5] Change summary')
    print(f'  Docs seen: {docs_seen}')
    print(f'  New docs: {docs_new}')
    print(f'  Changed docs: {docs_changed}')
    print(f'  Unchanged docs skipped: {docs_skipped}')
    print(f'  Removed docs: {len(removed_doc_ids)}')
    print(f'  Passages to upsert: {len(passages_to_upsert)}')

    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

    print('[Step 3/5] Applying deletions for stale points...')
    deleted_count = delete_points(client, stale_ids_to_delete) if stale_ids_to_delete else 0
    print(f'  Deleted points: {deleted_count}')

    print('[Step 4/5] Embedding + upsert...')
    upserted_count = 0
    if passages_to_upsert:
        texts = [p['text'] for p in passages_to_upsert]
        primary_vectors = embed_texts(EMBEDDING_MODEL, texts, BATCH_SIZE, NORMALIZE_EMBEDDINGS)
        sapbert_vectors = embed_texts(SAPBERT_MODEL, texts, BATCH_SIZE, NORMALIZE_EMBEDDINGS)

        ensure_collection(
            client,
            int(primary_vectors.shape[1]),
            int(sapbert_vectors.shape[1]),
        )
        upserted_count = upsert_points(
            client,
            passages_to_upsert,
            primary_vectors,
            sapbert_vectors,
        )
    else:
        print('  No new/updated passages to upsert.')

    print('[Step 5/5] Saving incremental state...')
    save_state(state_path, current_docs, INPUT_SOURCES)

    print('\n' + '=' * 72)
    print('Ingestion complete')
    print('=' * 72)
    print(f'Points upserted: {upserted_count}')
    print(f'Points deleted: {deleted_count}')
    print(f'Tracked documents: {len(current_docs)}')
    print(f'State written to: {state_path}')


In [ ]:
import gc
import os
from typing import Any, Dict, List, Tuple

EMBED_WRITE_CHUNK_SIZE = int(os.getenv('EMBED_WRITE_CHUNK_SIZE', '512'))
MAX_PASSAGES_PER_DOC = int(os.getenv('MAX_PASSAGES_PER_DOC', '0'))

print(f"[Memory] EMBED_WRITE_CHUNK_SIZE={EMBED_WRITE_CHUNK_SIZE}")
print(f"[Memory] MAX_PASSAGES_PER_DOC={MAX_PASSAGES_PER_DOC} (0 means unlimited)")

_model_cache: Dict[str, SentenceTransformer] = {}


def embed_texts(model_name: str, texts: List[str], batch_size: int, normalize: bool) -> np.ndarray:
    model = _model_cache.get(model_name)
    if model is None:
        print(f"[Embed] Loading model once: {model_name}")
        model = SentenceTransformer(model_name)
        _model_cache[model_name] = model

    print(f"[Embed] Encoding {len(texts)} passages...")
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    if normalize:
        vectors = l2_normalize(vectors)
        print("[Embed] L2 normalization applied")

    return vectors


_original_build_passages = build_passages


def build_passages(
    article: Dict[str, Any],
    source_site: str,
    source_type: str,
    source_file: str,
):
    doc_uid, passages, point_ids = _original_build_passages(article, source_site, source_type, source_file)

    if MAX_PASSAGES_PER_DOC > 0 and len(passages) > MAX_PASSAGES_PER_DOC:
        passages = passages[:MAX_PASSAGES_PER_DOC]
        point_ids = [p['point_id'] for p in passages]

    return doc_uid, passages, point_ids


def main() -> None:
    state_path = (PROJECT_ROOT / STATE_FILE).resolve()

    print('=' * 72)
    print(f"{SOURCE_SITE.upper()} -> Qdrant Cloud Incremental Ingestion (Colab, memory-safe)")
    print('=' * 72)
    print(f"[Config] Project root: {PROJECT_ROOT}")
    print(f"[Config] Qdrant URL: {QDRANT_URL}")
    print(f"[Config] Collection: {COLLECTION_NAME}")
    print(f"[Config] Fine-tuned model: {FINE_TUNE_MODEL_PATH}")
    print(f"[Config] State file: {state_path}")
    print(f"[Config] Embed-write chunk: {EMBED_WRITE_CHUNK_SIZE}")
    print(f"[Config] Max passages/doc: {MAX_PASSAGES_PER_DOC}")

    source_paths: List[Tuple[Path, str]] = []
    for src in INPUT_SOURCES:
        rel_path = src.get('path')
        source_type = src.get('source_type') or 'unknown'
        if not rel_path:
            continue

        abs_path = (PROJECT_ROOT / rel_path).resolve()
        if not abs_path.exists():
            print(f"  [Warn] Missing source, skipping: {abs_path}")
            continue

        source_paths.append((abs_path, source_type))
        print(f"  - {abs_path} ({source_type})")

    if not source_paths:
        raise RuntimeError(f"No valid {SOURCE_SITE} input sources found under PROJECT_ROOT.")

    state = load_state(state_path)
    previous_docs: Dict[str, Any] = state.get('docs', {})

    current_docs: Dict[str, Any] = {}
    passages_to_upsert: List[Dict[str, Any]] = []
    stale_ids_to_delete: List[str] = []

    docs_seen = 0
    docs_skipped = 0
    docs_new = 0
    docs_changed = 0

    print('[Step 1/5] Scanning and diffing source documents...')
    for source_path, source_type in source_paths:
        for doc, source_file in iter_source_docs(source_path, FILE_GLOB):
            docs_seen += 1
            if not isinstance(doc, dict) or not (doc.get('url') or doc.get('title')):
                continue
            if doc.get('error'):
                continue

            doc_uid = make_doc_uid(SOURCE_SITE, source_type, doc)
            fingerprint = canonical_hash(doc)
            previous = previous_docs.get(doc_uid)

            is_unchanged = (
                not FORCE_REPROCESS_ALL_DOCS
                and previous is not None
                and previous.get('fingerprint') == fingerprint
                and previous.get('primary_model') == EMBEDDING_MODEL
                and previous.get('sapbert_model') == SAPBERT_MODEL
            )

            if is_unchanged:
                current_docs[doc_uid] = previous
                docs_skipped += 1
                continue

            _, passages, point_ids = build_passages(doc, SOURCE_SITE, source_type, source_file)

            if previous is None:
                docs_new += 1
            else:
                docs_changed += 1
                if DELETE_OLD_PASSAGES_FOR_UPDATED_DOC:
                    stale_ids_to_delete.extend(previous.get('point_ids') or [])

            current_docs[doc_uid] = {
                'fingerprint': fingerprint,
                'point_ids': point_ids,
                'source_type': source_type,
                'primary_model': EMBEDDING_MODEL,
                'sapbert_model': SAPBERT_MODEL,
                'source_file': source_file,
                'updated_at_utc': utc_now_iso(),
            }
            passages_to_upsert.extend(passages)

    previous_doc_ids = set(previous_docs.keys())
    current_doc_ids = set(current_docs.keys())
    removed_doc_ids = sorted(previous_doc_ids - current_doc_ids)

    if removed_doc_ids and DELETE_REMOVED_DOCS:
        for doc_uid in removed_doc_ids:
            stale_ids_to_delete.extend(previous_docs.get(doc_uid, {}).get('point_ids') or [])

    print('[Step 2/5] Change summary')
    print(f"  Docs seen: {docs_seen}")
    print(f"  New docs: {docs_new}")
    print(f"  Changed docs: {docs_changed}")
    print(f"  Unchanged docs skipped: {docs_skipped}")
    print(f"  Removed docs: {len(removed_doc_ids)}")
    print(f"  Passages to upsert: {len(passages_to_upsert)}")

    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

    print('[Step 3/5] Applying deletions for stale points...')
    deleted_count = 0
    if stale_ids_to_delete:
        deleted_count = delete_points(client, stale_ids_to_delete)
    print(f"  Deleted points: {deleted_count}")

    print('[Step 4/5] Embedding + upsert in chunks...')
    upserted_count = 0
    if passages_to_upsert:
        total_chunks = (len(passages_to_upsert) + EMBED_WRITE_CHUNK_SIZE - 1) // EMBED_WRITE_CHUNK_SIZE
        collection_ready = False

        for chunk_idx in tqdm(range(total_chunks), desc='Embed+Upsert chunks'):
            start = chunk_idx * EMBED_WRITE_CHUNK_SIZE
            end = min(start + EMBED_WRITE_CHUNK_SIZE, len(passages_to_upsert))
            chunk_passages = passages_to_upsert[start:end]
            texts = [p['text'] for p in chunk_passages]

            primary_vectors = embed_texts(
                model_name=EMBEDDING_MODEL,
                texts=texts,
                batch_size=BATCH_SIZE,
                normalize=NORMALIZE_EMBEDDINGS,
            )
            sapbert_vectors = embed_texts(
                model_name=SAPBERT_MODEL,
                texts=texts,
                batch_size=BATCH_SIZE,
                normalize=NORMALIZE_EMBEDDINGS,
            )

            if not collection_ready:
                ensure_collection(
                    client=client,
                    primary_dim=int(primary_vectors.shape[1]),
                    sapbert_dim=int(sapbert_vectors.shape[1]),
                        )
                collection_ready = True

            upserted_count += upsert_points(
                client=client,
                passages=chunk_passages,
                primary_vectors=primary_vectors,
                sapbert_vectors=sapbert_vectors,
                )

            del texts
            del primary_vectors
            del sapbert_vectors
            del chunk_passages
            gc.collect()
    else:
        print('  No new/updated passages to upsert.')

    print('[Step 5/5] Saving incremental state...')
    save_state(state_path, current_docs, INPUT_SOURCES)

    print('\n' + '=' * 72)
    print('Ingestion complete')
    print('=' * 72)
    print(f"Points upserted: {upserted_count}")
    print(f"Points deleted: {deleted_count}")
    print(f"Tracked documents: {len(current_docs)}")
    print(f"State written to: {state_path}")


In [ ]:
import re
import uuid

HEX40_RE = re.compile(r"^[0-9a-f]{40}$")


def to_qdrant_point_id(pid):
    if isinstance(pid, int) and pid >= 0:
        return pid

    s = str(pid).strip()
    if not s:
        return str(uuid.uuid4())

    try:
        uuid.UUID(s)
        return s
    except Exception:
        pass

    # Legacy SHA1 hex IDs are converted to deterministic UUIDs.
    if HEX40_RE.fullmatch(s):
        return str(uuid.uuid5(uuid.NAMESPACE_OID, s))

    return str(uuid.uuid5(uuid.NAMESPACE_URL, s))


_original_build_passages_uuid = build_passages


def build_passages(article, source_site, source_type, source_file):
    doc_uid, passages, _ = _original_build_passages_uuid(article, source_site, source_type, source_file)

    for p in passages:
        p['point_id'] = to_qdrant_point_id(p.get('point_id'))

    point_ids = [p['point_id'] for p in passages]
    return doc_uid, passages, point_ids


_original_delete_points_uuid = delete_points


def delete_points(client, point_ids):
    normalized = []
    dropped = 0
    for pid in point_ids:
        if isinstance(pid, int) and pid >= 0:
            normalized.append(pid)
            continue

        s = str(pid).strip()
        if not s:
            dropped += 1
            continue

        try:
            uuid.UUID(s)
            normalized.append(s)
        except Exception:
            dropped += 1

    if dropped:
        print(f"[Qdrant] Skipped {dropped} stale IDs that were not valid UUID/int")

    return _original_delete_points_uuid(client, normalized)


print('[Patch] Point ID compatibility enabled: deterministic UUID conversion for upserts.')


In [ ]:
main()
